In [1]:
!nvidia-smi

Mon Aug 24 18:28:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
%%writefile graph.txt
6 8
0 1 4
0 2 2
1 2 1
1 3 5
2 3 8
2 4 10
3 4 2
4 5 3

Writing graph.txt


In [4]:
%%writefile sssp.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <climits>
#include <cuda.h>

using namespace std;

struct CSRGraph
{
    int V,E;
    vector<int> row_ptr;
    vector<int> col_idx;
    vector<int> weights;
};

CSRGraph readGraph(string filename)
{
    ifstream fin(filename);

    CSRGraph g;
    fin >> g.V >> g.E;

    vector<vector<pair<int,int>>> adj(g.V);

    int u,v,w;

    while(fin >> u >> v >> w)
    {
        adj[u].push_back({v,w});
    }

    g.row_ptr.resize(g.V+1);

    int edgeCount=0;

    for(int i=0;i<g.V;i++)
    {
        g.row_ptr[i]=edgeCount;

        for(auto &p:adj[i])
        {
            g.col_idx.push_back(p.first);
            g.weights.push_back(p.second);
            edgeCount++;
        }
    }

    g.row_ptr[g.V]=edgeCount;

    return g;
}

vector<int> cpuBellmanFord(CSRGraph &g,int source)
{
    vector<int> dist(g.V,INT_MAX);

    dist[source]=0;

    for(int k=0;k<g.V-1;k++)
    {
        for(int u=0;u<g.V;u++)
        {
            for(int e=g.row_ptr[u];
                e<g.row_ptr[u+1];
                e++)
            {
                int v=g.col_idx[e];
                int w=g.weights[e];

                if(dist[u]!=INT_MAX &&
                   dist[u]+w<dist[v])
                {
                    dist[v]=dist[u]+w;
                }
            }
        }
    }

    return dist;
}

__global__
void relaxKernel(
        int V,
        int *row_ptr,
        int *col_idx,
        int *weights,
        int *dist,
        int *changed)
{
    int u =
        blockIdx.x*blockDim.x +
        threadIdx.x;

    if(u>=V)
        return;

    int du = dist[u];

    if(du==INT_MAX)
        return;

    for(int e=row_ptr[u];
        e<row_ptr[u+1];
        e++)
    {
        int v = col_idx[e];

        int newDist =
            du + weights[e];

        int old =
            atomicMin(
                &dist[v],
                newDist);

        if(newDist < old)
            *changed = 1;
    }
}

vector<int> gpuBellmanFord(CSRGraph &g,int source)
{
    int V=g.V;
    int E=g.E;

    int *d_rowptr;
    int *d_colidx;
    int *d_weights;
    int *d_dist;
    int *d_changed;

    cudaMalloc(&d_rowptr,(V+1)*sizeof(int));
    cudaMalloc(&d_colidx,E*sizeof(int));
    cudaMalloc(&d_weights,E*sizeof(int));
    cudaMalloc(&d_dist,V*sizeof(int));
    cudaMalloc(&d_changed,sizeof(int));

    cudaMemcpy(
        d_rowptr,
        g.row_ptr.data(),
        (V+1)*sizeof(int),
        cudaMemcpyHostToDevice);

    cudaMemcpy(
        d_colidx,
        g.col_idx.data(),
        E*sizeof(int),
        cudaMemcpyHostToDevice);

    cudaMemcpy(
        d_weights,
        g.weights.data(),
        E*sizeof(int),
        cudaMemcpyHostToDevice);

    vector<int> dist(V,INT_MAX);
    dist[source]=0;

    cudaMemcpy(
        d_dist,
        dist.data(),
        V*sizeof(int),
        cudaMemcpyHostToDevice);

    int blockSize=256;
    int gridSize=
        (V+blockSize-1)/blockSize;

    for(int i=0;i<V-1;i++)
    {
        cudaMemset(
            d_changed,
            0,
            sizeof(int));

        relaxKernel<<<
            gridSize,
            blockSize>>>(
                V,
                d_rowptr,
                d_colidx,
                d_weights,
                d_dist,
                d_changed);

        cudaDeviceSynchronize();

        int changed;

        cudaMemcpy(
            &changed,
            d_changed,
            sizeof(int),
            cudaMemcpyDeviceToHost);

        if(changed==0)
            break;
    }

    cudaMemcpy(
        dist.data(),
        d_dist,
        V*sizeof(int),
        cudaMemcpyDeviceToHost);

    cudaFree(d_rowptr);
    cudaFree(d_colidx);
    cudaFree(d_weights);
    cudaFree(d_dist);
    cudaFree(d_changed);

    return dist;
}

int main()
{
    CSRGraph g =
        readGraph("graph.txt");

    auto cpu =
        cpuBellmanFord(g,0);

    auto gpu =
        gpuBellmanFord(g,0);

    cout<<"CPU Result\n";

    for(int i=0;i<g.V;i++)
        cout<<i<<" "<<cpu[i]<<endl;

    cout<<"\nGPU Result\n";

    for(int i=0;i<g.V;i++)
        cout<<i<<" "<<gpu[i]<<endl;

    return 0;
}

Writing sssp.cu


In [5]:
!nvcc sssp.cu -o sssp

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./sssp

CPU Result
0 0
1 4
2 2
3 9
4 11
5 14

GPU Result
0 0
1 4
2 2
3 9
4 11
5 14


In [8]:
%%writefile sssp2.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <climits>
#include <chrono>
#include <cuda.h>

using namespace std;

struct CSRGraph
{
    int V;
    int E;

    vector<int> row_ptr;
    vector<int> col_idx;
    vector<int> weights;
};

CSRGraph readGraph(string filename)
{
    ifstream fin(filename);

    CSRGraph g;

    fin >> g.V >> g.E;

    vector<vector<pair<int,int>>> adj(g.V);

    int u,v,w;

    while(fin >> u >> v >> w)
    {
        adj[u].push_back({v,w});
    }

    g.row_ptr.resize(g.V + 1);

    int edgeCount = 0;

    for(int i=0;i<g.V;i++)
    {
        g.row_ptr[i] = edgeCount;

        for(auto &p : adj[i])
        {
            g.col_idx.push_back(p.first);
            g.weights.push_back(p.second);
            edgeCount++;
        }
    }

    g.row_ptr[g.V] = edgeCount;

    return g;
}

vector<int> cpuBellmanFord(CSRGraph &g, int source)
{
    vector<int> dist(g.V, INT_MAX);

    dist[source] = 0;

    for(int iter=0; iter<g.V-1; iter++)
    {
        for(int u=0; u<g.V; u++)
        {
            for(int e=g.row_ptr[u];
                e<g.row_ptr[u+1];
                e++)
            {
                int v = g.col_idx[e];
                int w = g.weights[e];

                if(dist[u] != INT_MAX &&
                   dist[u] + w < dist[v])
                {
                    dist[v] = dist[u] + w;
                }
            }
        }
    }

    return dist;
}

__global__
void relaxKernel(
        int V,
        int *row_ptr,
        int *col_idx,
        int *weights,
        int *dist,
        int *changed)
{
    int u =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    if(u >= V)
        return;

    int du = dist[u];

    if(du == INT_MAX)
        return;

    for(int e=row_ptr[u];
        e<row_ptr[u+1];
        e++)
    {
        int v = col_idx[e];

        int newDist =
            du + weights[e];

        int old =
            atomicMin(&dist[v],
                      newDist);

        if(newDist < old)
            *changed = 1;
    }
}

vector<int> gpuBellmanFord(CSRGraph &g,
                           int source)
{
    int V = g.V;
    int E = g.E;

    int *d_rowptr;
    int *d_colidx;
    int *d_weights;
    int *d_dist;
    int *d_changed;

    cudaMalloc(&d_rowptr,
               (V+1)*sizeof(int));

    cudaMalloc(&d_colidx,
               E*sizeof(int));

    cudaMalloc(&d_weights,
               E*sizeof(int));

    cudaMalloc(&d_dist,
               V*sizeof(int));

    cudaMalloc(&d_changed,
               sizeof(int));

    cudaMemcpy(
        d_rowptr,
        g.row_ptr.data(),
        (V+1)*sizeof(int),
        cudaMemcpyHostToDevice);

    cudaMemcpy(
        d_colidx,
        g.col_idx.data(),
        E*sizeof(int),
        cudaMemcpyHostToDevice);

    cudaMemcpy(
        d_weights,
        g.weights.data(),
        E*sizeof(int),
        cudaMemcpyHostToDevice);

    vector<int> dist(V, INT_MAX);

    dist[source] = 0;

    cudaMemcpy(
        d_dist,
        dist.data(),
        V*sizeof(int),
        cudaMemcpyHostToDevice);

    int blockSize = 256;

    int gridSize =
        (V + blockSize - 1) /
        blockSize;

    for(int iter=0;
        iter<V-1;
        iter++)
    {
        cudaMemset(
            d_changed,
            0,
            sizeof(int));

        relaxKernel<<<
            gridSize,
            blockSize>>>(
                V,
                d_rowptr,
                d_colidx,
                d_weights,
                d_dist,
                d_changed);

        cudaDeviceSynchronize();

        int changed;

        cudaMemcpy(
            &changed,
            d_changed,
            sizeof(int),
            cudaMemcpyDeviceToHost);

        if(changed == 0)
            break;
    }

    cudaMemcpy(
        dist.data(),
        d_dist,
        V*sizeof(int),
        cudaMemcpyDeviceToHost);

    cudaFree(d_rowptr);
    cudaFree(d_colidx);
    cudaFree(d_weights);
    cudaFree(d_dist);
    cudaFree(d_changed);

    return dist;
}

int main()
{
    CSRGraph g =
        readGraph("graph.txt");

    auto cpu_start =
        chrono::high_resolution_clock::now();

    vector<int> cpuDist =
        cpuBellmanFord(g,0);

    auto cpu_end =
        chrono::high_resolution_clock::now();

    double cpu_time =
        chrono::duration<double,
        milli>(
            cpu_end -
            cpu_start).count();

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    vector<int> gpuDist =
        gpuBellmanFord(g,0);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float gpu_time;

    cudaEventElapsedTime(
        &gpu_time,
        start,
        stop);

    cout << "CPU Time : "
         << cpu_time
         << " ms\n";

    cout << "GPU Time : "
         << gpu_time
         << " ms\n";

    cout << "Speedup : "
         << cpu_time/gpu_time
         << "x\n\n";

    cout << "CPU Distances\n";

    for(int i=0;i<g.V;i++)
    {
        cout << i
             << " : "
             << cpuDist[i]
             << endl;
    }

    cout << "\nGPU Distances\n";

    for(int i=0;i<g.V;i++)
    {
        cout << i
             << " : "
             << gpuDist[i]
             << endl;
    }

    bool correct = true;

    for(int i=0;i<g.V;i++)
    {
        if(cpuDist[i] != gpuDist[i])
        {
            correct = false;
            break;
        }
    }

    if(correct)
        cout << "\nVerification Passed\n";
    else
        cout << "\nVerification Failed\n";

    return 0;
}

Writing sssp2.cu


In [9]:
!nvcc sssp2.cu -o sssp2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [10]:
!./sssp2

CPU Time : 0.00241 ms
GPU Time : 0.735328 ms
Speedup : 0.00327745x

CPU Distances
0 : 0
1 : 4
2 : 2
3 : 9
4 : 11
5 : 14

GPU Distances
0 : 0
1 : 4
2 : 2
3 : 9
4 : 11
5 : 14

Verification Passed
